# D&D 5E Rules Agent — Week 3 Capstone: Adding Agents to Your Project

Week 1 ended with a RAG pipeline in `02-rag.ipynb`: `search` → `build_prompt` → `llm` → `rag`.
It works, but it is fixed: exactly one retrieval call, no way to read more of a book when a
passage stops mid-rule, no idea what it knows, and no memory of anything you tell it.

This notebook turns that pipeline into an **agent with tools** — the model decides which tool
to call, reads the result, and calls the next tool if it needs more. Same data, same index,
same citations; new control flow.

## The tools (the plan from Question 1)

| tool | what it does and when | in → out |
|---|---|---|
| `search_rules` | search the rulebook index — the week-1 `search` function as a tool. First call for every rules/spell/monster question | `query`, `num_results` → passages with book title, filename, `start` offset and text |
| `read_passage` | pull a longer window out of one book when a search passage is cut off mid-rule | `filename`, `start`, `length` → raw text |
| `list_sources` | answer "which books do you actually know?" and check scope before declaring a question unanswerable | – → title, filename, size per book |
| `save_note` | remember a house rule / campaign note across sessions (writes `data/notes.json`) | `topic`, `note` → confirmation |

* **Essential:** `search_rules` (it *is* the week-1 `search`).
* **Nice to have:** `read_passage` (context when a hit is truncated), `list_sources`
  (scoping), `save_note` (the write tool — the assistant becomes stateful).

Pattern follows the course reference project `documentation-agent/tools.py` +
`documentation-agent/doc_agent.py`: small focused tools on a class, one agent that receives
them, and a print of every tool call so the run is inspectable.

In [1]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")  # keep the notebook output readable

from minsearch import Index
from pydantic_ai import Agent

# Works whether Jupyter starts in the repo root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data"
CACHE = DATA_DIR / "documents.json"
NOTES_FILE = DATA_DIR / "notes.json"

documents = json.loads(CACHE.read_text())
print(f"project root : {ROOT}")
print(f"books        : {len(documents)}")
for doc in documents:
    print(f"  - {doc['title']} ({len(doc['content']):,} chars)")

project root : /home/george/repos/dnd-rag-assistant
books        : 10
  - Player's Handbook (D&D 5E) (1,203,595 chars)
  - Dungeon Master's Guide (D&D 5E) (1,190,796 chars)
  - Monster Manual (D&D 5E) (1,192,376 chars)
  - Eberron: Rising from the Last War (D&D 5E) (1,350,235 chars)
  - Xanathar's Guide to Everything (D&D 5E) (731,647 chars)
  - Tasha's Cauldron of Everything (D&D 5E) (631,191 chars)
  - Volo's Guide to Monsters (D&D 5E) (950,679 chars)
  - Mordenkainen's Tome of Foes (D&D 5E) (1,030,339 chars)
  - Diario di Campagna Eberron - Prologo (Giorni 1/7, Sessioni 1/3) (87,109 chars)
  - Trascrizione Schede Personaggi D&D (7,846 chars)


## 1. Same index as week 1

Rebuild the chunk index from the cached `data/documents.json` (the PDF extraction itself is
skipped — that cache was produced by `02-rag.ipynb`).

In [2]:
from gitsource import chunk_documents

EXCLUDE_FROM_CHUNKING = {
    "5208a1d4-62b5-495b-8905-382e6f8f946d.pdf",
    "Trascrizione Schede Personaggi D&D.pdf",
}

document_chunks = []
for doc in documents:
    if doc["filename"] in EXCLUDE_FROM_CHUNKING:
        document_chunks.append({**doc, "start": 0, "chunk_id": 0})
    else:
        for i, chunk in enumerate(chunk_documents([doc], size=3000, step=1500)):
            chunk["chunk_id"] = i
            document_chunks.append(chunk)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"],
)
index.fit(document_chunks)

print(f"indexed {len(document_chunks)} chunks from {len(documents)} books")

indexed 5517 chunks from 10 books


## 2. The retrieval function, now with offsets

`search_passages` is the week-1 `search` with one addition: every passage carries the
`start` offset of its first chunk, so the agent can hand that offset straight to
`read_passage` and widen the window.

In [3]:
def merge_chunks(chunks):
    """Merge overlapping/adjacent chunks from the same document into one passage."""
    chunks = sorted(chunks, key=lambda c: c["start"])
    merged = chunks[0]["content"]
    end = chunks[0]["start"] + len(merged)

    for chunk in chunks[1:]:
        chunk_end = chunk["start"] + len(chunk["content"])
        if chunk_end <= end:
            continue
        overlap = max(end - chunk["start"], 0)
        if overlap == 0:
            merged += "\n...\n"
        merged += chunk["content"][overlap:]
        end = chunk_end

    return merged


BOOST = {"title": 3.0, "description": 2.0, "content": 1.0}


def search_passages(query, num_results=3):
    """Search chunks, merge adjacent hits per book, return passages."""
    hits = index.search(query, num_results=num_results * 3, boost_dict=BOOST)

    grouped = {}
    for hit in hits:
        grouped.setdefault(hit["filename"], []).append(hit)

    passages = []
    for filename, chunks in grouped.items():
        chunks = sorted(chunks, key=lambda c: c["start"])
        content = merge_chunks(chunks) if len(chunks) > 1 else chunks[0]["content"]
        passages.append(
            {
                "title": chunks[0]["title"],
                "filename": filename,
                "start": chunks[0]["start"],
                "content": content,
            }
        )

    return passages[:num_results]

## 3. The tools

Four small, focused tools on one class — one per job, no kitchen-sink tool. Each call appends
to `self.trace` so we can print exactly what the agent did (the reference project does the
same through PydanticAI's event stream; logging inside the tools is the sync equivalent and
keeps the notebook readable).

In [4]:
class DndTools:
    """The tools the agent can call, plus a trace of the calls."""

    def __init__(self, documents):
        self.documents = documents
        self.by_filename = {d["filename"]: d for d in documents}
        self.trace = []

    # --- read/search -----------------------------------------------------

    def search_rules(self, query: str, num_results: int = 3) -> list:
        """Search the D&D 5E rulebook index for a rules, spell, monster or
        character question. Returns passages: book `title`, `filename`, the
        character offset `start` in that book, and the passage `content`.
        Call this first for any question about the rules."""
        passages = search_passages(query, num_results=num_results)
        self.trace.append(
            {"tool": "search_rules", "args": {"query": query, "num_results": num_results},
             "result": f"{len(passages)} passages from "
                       + ", ".join(sorted({p['title'] for p in passages}))}
        )
        return passages

    def read_passage(self, filename: str, start: int, length: int = 4000) -> str:
        """Read `length` characters of one book starting at character offset
        `start`. Use it when a search passage cuts off in the middle of a rule.
        `filename` and `start` must come from a search_rules result."""
        doc = self.by_filename.get(filename)
        if doc is None:
            return f"error: no book named '{filename}' — call list_sources first"
        self.trace.append({"tool": "read_passage",
                           "args": {"filename": filename, "start": start, "length": length},
                           "result": f"{min(length, len(doc['content']) - start)} chars of {doc['title']}"})
        return doc["content"][start:start + length]

    # --- scope and memory ------------------------------------------------

    def list_sources(self) -> list:
        """List the books in the knowledge base (title, filename, size). Use it
        to answer 'what do you know about' questions and to check scope."""
        self.trace.append({"tool": "list_sources", "args": {},
                           "result": f"{len(self.documents)} books"})
        return [
            {"title": d["title"], "filename": d["filename"], "characters": len(d["content"])}
            for d in self.documents
        ]

    def save_note(self, topic: str, note: str) -> str:
        """Save a campaign note or house rule to data/notes.json so that later
        sessions know about it. Use when the user asks you to remember, note
        down or record something."""
        notes = json.loads(NOTES_FILE.read_text()) if NOTES_FILE.exists() else []
        notes.append({"topic": topic, "note": note})
        NOTES_FILE.write_text(json.dumps(notes, indent=2, ensure_ascii=False))
        self.trace.append({"tool": "save_note", "args": {"topic": topic},
                           "result": f"note #{len(notes)} stored"})
        return f"saved note #{len(notes)} '{topic}' to {NOTES_FILE.relative_to(ROOT)}"


tools = DndTools(documents)
print("tools:", [name for name in dir(tools) if not name.startswith("_") and name not in ("documents", "by_filename", "trace")])

tools: ['list_sources', 'read_passage', 'save_note', 'search_rules']


## 4. The agent

PydanticAI, same shape as the course's `doc_agent.py`: one `Agent`, an `instructions` string
that is specific to this project (not "you are a helpful assistant"), and the four tools.

In [5]:
INSTRUCTIONS = """
You are a Dungeons & Dragons 5th Edition rules assistant with four tools:
search_rules, read_passage, list_sources and save_note. Use the tools; do not
answer rules questions from memory.

How to work:
1. For any rules, spell, monster or character question, start with one
   search_rules call. Reformulate the query once if the first search is empty.
2. If a passage stops in the middle of a rule, use read_passage with that
   passage's filename and start offset to read more of the same book. Do not
   read blindly: only use offsets you got from search_rules.
3. For "which books do you know" / "can you answer X" questions, call
   list_sources before answering, and be honest about what is not in them.
4. When the user asks you to remember, note down or record something
   (usually a house rule or a campaign note), call save_note with a short
   topic and the note.

Answer rules:
- Cite the book for every claim, e.g. (Player's Handbook, D&D 5E), using the
  title of the passage you actually read. Never invent a rule, a number or a
  page that is not in the retrieved text.
- If the tools do not contain the answer, say you don't know and name the book
  that would most likely contain it.
- Answer in the language the user writes in (English or Italian).
- For character-building questions give the concrete rules (prerequisites,
  slots, action economy), not general advice.
- If a question is about another game, or is not about D&D at all, say so in
  one short sentence and offer to answer a D&D question instead. Do not search
  more than once for an obviously unrelated question.
""".strip()


agent = Agent(
    "openai-responses:gpt-4o-mini",
    name="dnd-rules-agent",
    instructions=INSTRUCTIONS,
    tools=[tools.search_rules, tools.read_passage, tools.list_sources, tools.save_note],
)

print("agent ready:", agent.name, "| model:", agent.model)

agent ready: dnd-rules-agent | model: OpenAIResponsesModel()


## 5. Run it and watch the tools

`ask()` clears the trace, runs the agent, prints every tool call with its arguments and its
result, and returns the final answer. It is `async` and called with a top-level `await` —
PydanticAI's sync API can't run inside a live Jupyter event loop.

In [6]:
async def ask(question, agent=agent, message_history=None):
    tools.trace.clear()
    result = await agent.run(question, message_history=message_history)

    print(f"QUESTION: {question}")
    print(f"\nTOOL CALLS ({len(tools.trace)}):")
    for i, call in enumerate(tools.trace, 1):
        print(f"  {i}. {call['tool']}({json.dumps(call['args'], ensure_ascii=False)})")
        print(f"     -> {call['result']}")
    print(f"\nANSWER:\n{result.output}")
    return result.output, result.all_messages()

### Test 1 — the main tool path (search)

A normal mid-session rules question.

In [7]:
answer, messages = await ask("How does an opportunity attack work? What triggers it and what does it cost?")

QUESTION: How does an opportunity attack work? What triggers it and what does it cost?

TOOL CALLS (3):
  1. search_rules({"query": "opportunity attack", "num_results": 3})
     -> 2 passages from Dungeon Master's Guide (D&D 5E), Player's Handbook (D&D 5E)
  2. read_passage({"filename": "D&D 5E - Player's Handbook.pdf", "start": 244500, "length": 4000})
     -> 4000 chars of Player's Handbook (D&D 5E)
  3. read_passage({"filename": "D&D 5E - Player's Handbook.pdf", "start": 244484, "length": 4000})
     -> 4000 chars of Player's Handbook (D&D 5E)

ANSWER:
An opportunity attack occurs when a hostile creature that you can see moves out of your reach. Here are the key details:

1. **Trigger**: You can make an opportunity attack when a creature you can see leaves your reach. 

2. **Cost**: To make an opportunity attack, you use your reaction. This attack interrupts the creature's movement and happens right before it leaves your reach.

3. **Action Type**: The opportunity attack is a melee 

### Test 2 — a different tool path (scope + write)

Two questions that should *not* end in `search_rules`: one about scope, one that must be
written to disk.

In [8]:
answer, messages = await ask("Which books do you actually have indexed? I want to know what you can answer from.")

QUESTION: Which books do you actually have indexed? I want to know what you can answer from.

TOOL CALLS (1):
  1. list_sources({})
     -> 10 books

ANSWER:
I have access to the following Dungeons & Dragons 5th Edition books:

1. **Player's Handbook (D&D 5E)**
2. **Dungeon Master's Guide (D&D 5E)**
3. **Monster Manual (D&D 5E)**
4. **Eberron: Rising from the Last War (D&D 5E)**
5. **Xanathar's Guide to Everything (D&D 5E)**
6. **Tasha's Cauldron of Everything (D&D 5E)**
7. **Volo's Guide to Monsters (D&D 5E)**
8. **Mordenkainen's Tome of Foes (D&D 5E)**
9. **Diario di Campagna Eberron - Prologo (Giorni 1/7, Sessioni 1/3)**
10. **Trascrizione Schede Personaggi D&D**

Feel free to ask questions about the rules, spells, monsters, or character details from these sources!


In [9]:
# Start from a clean slate so re-running the notebook doesn't duplicate the note.
NOTES_FILE.unlink(missing_ok=True)

answer, messages = await ask(
    "Remember this house rule for our campaign: flanking gives advantage on melee attack rolls. "
    "Write it down so you don't forget it."
)

print("\nnotes on disk:", json.loads(NOTES_FILE.read_text()))

QUESTION: Remember this house rule for our campaign: flanking gives advantage on melee attack rolls. Write it down so you don't forget it.

TOOL CALLS (1):
  1. save_note({"topic": "Flanking House Rule"})
     -> note #1 stored

ANSWER:
I've noted your house rule: flanking gives advantage on melee attack rolls. If you have any more rules or notes to add, just let me know!

notes on disk: [{'topic': 'Flanking House Rule', 'note': 'Flanking gives advantage on melee attack rolls.'}]


### Test 3 — an edge case (outside its scope)

A question a D&D 5E rules assistant should *not* answer: another game's rules.

In [10]:
answer, messages = await ask("What are the best feats for a Pathfinder 2e alchemist?")

QUESTION: What are the best feats for a Pathfinder 2e alchemist?

TOOL CALLS (0):

ANSWER:
I'm focused on Dungeons & Dragons 5th Edition rules, so I can't provide specific advice on Pathfinder 2e. If you have any D&D-related questions, feel free to ask!


## 6. What this bought us over week 1

- The model **chooses** the tool: `search_rules` for rules, `list_sources` for scope,
  `save_note` for memory — one pipeline in `02-rag.ipynb` could only ever do the first.
- `read_passage` lets it fix a truncated passage instead of guessing the missing half of a
  rule.
- `save_note` makes the assistant stateful: `data/notes.json` survives the session, so the
  house rule written in test 2 is available to the next run.
- The trace makes failures debuggable — when the wrong tool is called, you see exactly which
  argument misled it.

Next: multi-turn conversation (`message_history` is already threaded through `ask`), an eval
set of rules questions with/without the agent, and reading the notes back into the prompt.